The dataset has these relevant columns: instruction, category, intent, response. For this basic pipeline, treat response as your document (the knowledge you retrieve) and instruction as example user queries (useful later for testing).

In [1]:
from datasets import load_dataset

ds = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset")["train"]


df = ds.to_pandas()[["response", "intent", "category"]]

README.md:   0%|          | 0.00/11.9k [00:00<?, ?B/s]

Bitext_Sample_Customer_Support_Training_(…): reconstructing file:   0%|          |  0.00B / 19.2MB            

Bitext_Sample_Customer_Support_Training_(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/26872 [00:00<?, ? examples/s]

In [2]:
df.head()

,response,intent,category
0,I've understood you have a question regarding ...,cancel_order,ORDER
1,I've been informed that you have a question ab...,cancel_order,ORDER
2,I can sense that you're seeking assistance wit...,cancel_order,ORDER
3,I understood that you need assistance with can...,cancel_order,ORDER
4,I'm sensitive to the fact that you're facing f...,cancel_order,ORDER


In [11]:
df['response'][0]

"I've understood you have a question regarding canceling order {{Order Number}}, and I'm here to provide you with the information you need. Please go ahead and ask your question, and I'll do my best to assist you."

text Preprcessing

In [12]:
import re

def clean_text(text: str) -> str:
    text = re.sub(r"\{\{.*?\}\}", "", text)      # remove template placeholders
    text = re.sub(r"<.*?>", "", text)              # strip stray HTML tags
    text = re.sub(r"\s+", " ", text).strip()        # collapse whitespace
    return text

df["response"] = df["response"].apply(clean_text)
df = df[df["response"].str.len() > 10]              # drop empty/too-short rows
df = df.drop_duplicates(subset="response")            # dedupe identical answers
df = df.reset_index(drop=True)

In [13]:
len(df)

26869

chunking

In [14]:
def chunk_text(text: str, max_words: int = 120) -> list[str]:
    words = text.split()
    if len(words) <= max_words:
        return [text]
    return [" ".join(words[i:i+max_words]) for i in range(0, len(words), max_words)]

chunks = []
chunk_metadata = []
for idx, row in df.iterrows():
    for c in chunk_text(row["response"]):
        chunks.append(c)
        chunk_metadata.append({"source_id": idx, "intent": row["intent"], "category": row["category"]})

In [32]:
import pickle

with open("/content/chunks.pkl", "wb") as f:
    pickle.dump({"chunks": chunks, "metadata": chunk_metadata}, f)

In [33]:
# load later
with open("/content/chunks.pkl", "rb") as f:
    data = pickle.load(f)
chunks = data["chunks"]
chunk_metadata = data["metadata"]

In [15]:
len(chunks)

34586

In [11]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")  # 384-dim

embeddings = model.encode(
    chunks,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True   # important: lets Qdrant use cosine similarity cleanly
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/541 [00:00<?, ?it/s]

In [13]:
embeddings.shape

(34586, 384)

In [14]:
type(embeddings)

numpy.ndarray

need to load -> embeddings only as we have computed it and saved it

In [2]:
import numpy as np

In [17]:
np.save("rag_embeddings.npy",embeddings)


In [4]:
embeddings = np.load("/content/rag_embeddings.npy")
embeddings.shape

(34586, 384)

In [5]:
!pip install qdrant-client -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.2/396.2 kB 12.8 MB/s eta 0:00:00


In [17]:
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct


client = QdrantClient(path="/content/qdrant_db")

client.recreate_collection(
    collection_name="support_kb",
    vectors_config=VectorParams(size=384, distance=Distance.COSINE)
)

/tmp/ipykernel_1398/3932031301.py:8: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


True

In [18]:
points = [
    PointStruct(
        id=i,
        vector=embeddings[i].tolist(),
        payload={"text": chunks[i], **chunk_metadata[i]}
    )
    for i in range(len(chunks))
]


batch_size = 512
for i in range(0, len(points), batch_size):
    client.upsert(collection_name="support_kb", points=points[i:i+batch_size])

/tmp/ipykernel_1398/1809420228.py:13: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Current collection contains 20480 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  client.upsert(collection_name="support_kb", points=points[i:i+batch_size])


In [22]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
model

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)

TOP k retrieval

In [23]:
def retrieve(query: str, k: int = 5):
    query_vec = model.encode([query], normalize_embeddings=True)[0]
    results = client.query_points(
        collection_name="support_kb",
        query=query_vec.tolist(),
        limit=k
    )
    return [point.payload["text"] for point in results.points]

In [24]:
print(retrieve("how do I cancel my order?"))

["I've understood you have a question about canceling order . How can I assist you?", "I acknowledge that you're unsure about how to cancel order , and I'm here to guide you through the process. Rest assured, I'll provide you with the necessary steps to cancel your order smoothly. 1. Access Your Account: Begin by logging into your using your credentials. 2. Locate Your Order: Once you're logged in, navigate to the '' or '' section to find the order in question. 3. Choose the Order: Identify order from the list and select it to view its details. 4. Initiate Cancellation: Look for the option labeled '' associated with the purchase and click on it to begin the cancellation process. 5. Follow Instructions: Depending on the platform, you might be prompted to confirm the cancellation", "I'm conscious of the reality that you're unsure about how to cancel order . I'm here to assist you throughout the process. To cancel your order, please follow these steps: 1. Access Your Account: Log in to ou

In [25]:
!pip install transformers accelerate bitsandbytes -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 23.0 MB/s eta 0:00:00


In [27]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

qwen_model_name = "Qwen/Qwen3-1.7B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(qwen_model_name)
qwen = AutoModelForCausalLM.from_pretrained(
    qwen_model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

model.safetensors.index.json:   0%|          | 0.00/25.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [30]:
def generate_answer(query: str, k: int = 5) -> str:
    context_chunks = retrieve(query, k=k)
    context = "\n".join(f"- {c}" for c in context_chunks)

    prompt = f"""Answer the user's question using only the context below.
If the context doesn't contain the answer, say you don't know.

Context:
{context}

Question: {query}
Answer:"""

    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        enable_thinking=False,
        return_dict=True,          # <-- forces a proper dict with input_ids + attention_mask
        return_tensors="pt"
    ).to(qwen.device)

    output = qwen.generate(**inputs, max_new_tokens=256, temperature=0.3)   # <-- unpack with **
    return tokenizer.decode(
        output[0][inputs["input_ids"].shape[-1]:],   # <-- index into input_ids, not inputs directly
        skip_special_tokens=True
    )

In [31]:
print(generate_answer("How do I cancel my order?"))

I can help you cancel your order. Please follow these steps: 

1. Sign in to your account using your credentials. 
2. Navigate to the "Orders" or "Order History" section of your account. 
3. Locate the specific order with the order number and click on it to view the details. 
4. Look for the option labeled "Cancel" or similar and click on it to start the cancellation process. 
5. Follow the prompts and provide any required information to complete the cancellation. 

If you encounter any difficulties or have further questions, our dedicated customer support team is available to assist you.
